![image](https://raw.githubusercontent.com/IBM/watsonx-ai-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use watsonx and `ibm/granite-4-h-micro` to generate code based on instruction

#### Disclaimers

- Use only Projects and Spaces that are available in watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate support for code generation in watsonx.ai. It introduces commands for defining prompt and model testing.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The goal of this notebook is to demonstrate how to generate code using `ibm/granite-4-h-micro` model based on instruction provided by the user.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Foundation models on IBM watsonx.ai](#Foundation-models-on-IBM-watsonx.ai)
3. [Generate code based on instruction](#Generate-code-based-on-instruction)
4. [Generated code testing](#Generated-code-testing)
5. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U ibm-watsonx-ai | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

### Working with projects

First of all, you need to create a project that will be used for your work. If you do not have a project created already, follow the steps below:

- Open IBM Cloud Pak® main page
- Click all projects
- Create an empty project
- Copy `project_id` from url and paste it below

**Action**: Assign project ID below

In [4]:
import os

try:
    project_id = os.environ["PROJECT_ID"]
except KeyError:
    project_id = input("Please enter your project_id (hit enter): ")

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials, project_id)

<a id="Foundation-models-on-IBM-watsonx.ai"></a>
## Foundation models on IBM watsonx.ai

#### List available models

In [6]:
for model in client.foundation_models.ChatModels:
    print(f"- {model}")

- ibm/granite-4-h-micro
- ibm/ibm-defense-3-3-8b-instruct
- magistral-small-2509
- meta-llama/llama-3-2-1b-instruct
- ministral-8b-instruct-2512
- mistralai/mistral-small-3-2-24b-instruct-2506


You need to specify `model_id` that will be used for inferencing:

In [7]:
model_id = client.foundation_models.ChatModels.GRANITE_4_H_MICRO

### Defining the model parameters

You might need to adjust model `parameters` for different models or tasks, to do so please refer to <a href="https://ibm.github.io/watsonx-ai-python-sdk/fm_model.html#metanames.GenTextParamsMetaNames" target="_blank" rel="noopener no referrer">documentation</a>.

In [8]:
from ibm_watsonx_ai.metanames import GenChatParamsMetaNames

parameters = {
    GenChatParamsMetaNames.TEMPERATURE: 0,
}

### Initialize the model
Initialize the model inference with previously set parameters.

In [9]:
from ibm_watsonx_ai.foundation_models import ModelInference

model = ModelInference(model_id=model_id, params=parameters, api_client=client)

### Get model details

In [10]:
model.get_details()

{'model_id': 'ibm/granite-4-h-micro',
 'label': 'granite-4-h-micro',
 'provider': 'IBM',
 'source': 'IBM',
 'functions': [{'id': 'text_chat'}],
 'short_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets.',
 'long_description': 'Granite-4.0-H-micro is a 30B parameter long-context instruct model finetuned from Granite-4.0-H-micro-Base using a combination of open source instruction datasets with permissive license and internally collected synthetic datasets. This model is developed using a diverse set of techniques with a structured chat format, including supervised finetuning, model alignment using reinforcement learning, and model merging. Granite 4.0 instruct models feature improved instruction following (IF) and tool-calling capabilities, making them more effective in enterprise applications.'

<a id="Generate-code-based-on-instruction"></a>
## Generate code based on instruction

Define instructions for the model with at least one example.

In [11]:
messages = [
    {
        "role": "system",
        "content": "Using the directions below, generate Python code for the given task. Do not provide any other text.",
    },
    {
        "role": "user",
        "content": "Write a Python function that prints 'Hello World!' string 'n' times.",
    },
    {
        "role": "assistant",
        "content": "\n".join(
            [
                "def print_n_times(n):",
                "    for i in range(n):",
                "        print('Hello World!')",
            ]
        ),
    },
    {
        "role": "user",
        "content": (
            "Write a Python function, which generates sequence of prime numbers. "
            "The function 'primes' will take the argument 'n', an int. "
            "It will return a list which contains all primes less than 'n'."
        ),
    },
]

### Generate the code using `ibm/granite-4-h-micro` model

Inference the model to generate the code, according to provided instruction.

In [12]:
result = model.chat(messages)
code_as_text = result["choices"][0]["message"]["content"]
print(code_as_text)

def primes(n):
    primes = []
    for possiblePrime in range(2, n):
        isPrime = True
        for num in range(2, int(possiblePrime ** 0.5) + 1):
            if possiblePrime % num == 0:
                isPrime = False
                break
        if isPrime:
            primes.append(possiblePrime)
    return primes


<a id="Generated-code-testing"></a>
## Generated code testing

Use generated code to make it as function.

**Note**: Before executing this line, make sure the model's output visible above doesn't contain any malicious instructions.

In [13]:
exec(code_as_text)

Define the number 'n' for which the primes() function should process prime numbers.

In [14]:
n = 25

Test and run the generated function.

In [15]:
primes(n)

[2, 3, 5, 7, 11, 13, 17, 19, 23]

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!
 
You learned how to generate code based on instruction with `ibm/granite-4-h-micro` in watsonx.ai. 
 
Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors

**Mateusz Szewczyk**, Software Engineer at watsonx.ai.

Copyright © 2023-2026 IBM. This notebook and its source code are released under the terms of the MIT License.